# Phishing Datasets Linear Regression Analysis
This notebook processes two datasets:
1. `StealthPhisher2025(2).csv`
2. `PhiUSIIL_Phishing_URL_Dataset(1).csv`

It applies linear regression, adding polynomial terms, interaction terms, evaluating multicollinearity with Variance Inflation Factor (VIF), and processing both categorical and continuous features.

In [9]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

## 1. Defining the Processing Function
We create a function that takes a dataset path, its target column, and lists of columns to drop. It performs the necessary steps: handling categorical/continuous variables, creating polynomial/interaction terms, computing VIF, and running linear regression.

In [12]:
def analyze_dataset(file_path, target_col, cols_to_drop):
    print(f"\n{'='*50}\nAnalyzing {file_path}\n{'='*50}")
    
    # 1. Load Data
    df = pd.read_csv(file_path)
    
    # Drop irrelevant or high cardinality columns (like URL strings)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    
    # Handle missing values 
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in df.columns:
        if col in numeric_cols:
            df[col].fillna(df[col].median(), inplace=True) 
        else:
            if not df[col].mode().empty:
                df[col].fillna(df[col].mode()[0], inplace=True) 
            
    # Sample down to 5000 rows to prevent MemoryErrors with polynomial generation
    if len(df) > 5000:
        df = df.sample(5000, random_state=42)
        print("Sampled down to 5000 rows for memory limits.")
            
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    if y.dtype == 'object' or y.dtype == 'string':
        encoder = LabelEncoder()
        y = encoder.fit_transform(y)
    
    
    
    # Identify Categorical and Continuous Features
    categorical_features = [col for col in X.columns if col not in numeric_cols or X[col].nunique() < 10]
    continuous_features = [col for col in X.columns if col not in categorical_features]
    
    # 2. Preprocessing: Encoding and Scaling
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])
    
    X_processed = preprocessor.fit_transform(X)
    
    # Get feature names after preprocessing
    cat_encoder = preprocessor.named_transformers_['cat']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
    feature_names = continuous_features + list(cat_feature_names)
    
    # Limit features to the most variant ones to keep VIF and Polynomials manageable
    X_df = pd.DataFrame(X_processed, columns=feature_names)
    X_df = X_df.loc[:, X_df.var() > 0.05] 
    
    if X_df.shape[1] > 30:
        print(f"Reducing features from {X_df.shape[1]} to 30 for polynomial/interaction terms.")
        top_features = X_df.var().sort_values(ascending=False).head(30).index
        X_df = X_df[top_features]
        
    # 3. Variance Inflation Factor (Multicollinearity)
    print("\n--- Variance Inflation Factor (VIF) ---")
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X_df.columns
    vif_data["VIF"] = [variance_inflation_factor(X_df.values, i) for i in range(len(X_df.columns))]
    print(vif_data.sort_values(by="VIF", ascending=False).head(5))
    
    # 4. Polynomial and Interaction Terms
    print("\n--- Generating Polynomial and Interaction Terms ---")
    poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
    X_poly = poly.fit_transform(X_df)
    
    print(f"Original feature count: {X_df.shape[1]}")
    print(f"Feature count after Polynomials and Interactions: {X_poly.shape[1]}")
    
    # 5. Linear Regression
    X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("\n--- Linear Regression Results ---")
    print(f"Mean Squared Error: {mse:.4f}")
    print(f"R-squared: {r2:.4f}")
    print("==================================================\n")

## 2. Analysis of `StealthPhisher2025(2).csv`

In [14]:
stealth_file = 'StealthPhisher2025(2).csv'
cols_to_drop_stealth = ['URL', 'Domain', 'TLD', 'HasTitle', 'Title']
analyze_dataset(stealth_file, 'Label', cols_to_drop_stealth)


Analyzing StealthPhisher2025(2).csv
Sampled down to 5000 rows for memory limits.
Reducing features from 66 to 30 for polynomial/interaction terms.

--- Variance Inflation Factor (VIF) ---
           Feature          VIF
23     LengthOfURL  2625.101085
25  LetterCntInURL  1775.739140
1    DigitCntInURL   162.874941
5   URLLetterRatio    93.130175
15   URLDigitRatio    66.196809

--- Generating Polynomial and Interaction Terms ---
Original feature count: 30
Feature count after Polynomials and Interactions: 495

--- Linear Regression Results ---
Mean Squared Error: 0.1552
R-squared: 0.3791



## 3. Analysis of `PhiUSIIL_Phishing_URL_Dataset(1).csv`

In [15]:
phiusiil_file = 'PhiUSIIL_Phishing_URL_Dataset(1).csv'
cols_to_drop_phiusiil = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
analyze_dataset(phiusiil_file, 'label', cols_to_drop_phiusiil)


Analyzing PhiUSIIL_Phishing_URL_Dataset(1).csv
Sampled down to 5000 rows for memory limits.
Reducing features from 59 to 30 for polynomial/interaction terms.

--- Variance Inflation Factor (VIF) ---
               Feature           VIF
27  HasCopyrightInfo_1  1.000800e+15
26  HasCopyrightInfo_0  1.000800e+15
29      HasSocialNet_1  1.000800e+15
28      HasSocialNet_0  1.000800e+15
8            URLLength  1.074143e+03

--- Generating Polynomial and Interaction Terms ---
Original feature count: 30
Feature count after Polynomials and Interactions: 495

--- Linear Regression Results ---
Mean Squared Error: 0.0095
R-squared: 0.9615

